In [2]:
import pandas as pd

main_df = pd.read_csv(r"C:\Users\20245179\OneDrive - TU Eindhoven\WP2_Simulation_Tool\data\final_data_cleaned_with_author_names.csv")
sentiment_df = pd.read_csv(r"C:\Users\20245179\OneDrive - TU Eindhoven\WP2_Simulation_Tool\data\Real_Sentiments.csv")
intent_df = pd.read_csv(r"C:\Users\20245179\OneDrive - TU Eindhoven\WP2_Simulation_Tool\data\intent_predictions_fine_tuned_4_intents_real_data_distribution_majority_voting_3_models.csv")

In [4]:
# Examine the structure of each dataframe
print("Main DataFrame Info:")
print(f"Shape: {main_df.shape}")
print(f"Columns: {list(main_df.columns)}")
print("\nSentiment DataFrame Info:")
print(f"Shape: {sentiment_df.shape}")
print(f"Columns: {list(sentiment_df.columns)}")
print("\nIntent DataFrame Info:")
print(f"Shape: {intent_df.shape}")
print(f"Columns: {list(intent_df.columns)}")

# Check for the specific columns we need to merge
print("\n" + "="*60)
print("COLUMN VERIFICATION:")
print("="*60)

# Check text column in main_df
if 'text' in main_df.columns:
    print("✓ 'text' column found in main_df")
else:
    print("✗ 'text' column NOT found in main_df")
    print(f"Available columns: {list(main_df.columns)}")

# Check Comments column in intent_df
if 'Comments' in intent_df.columns:
    print("✓ 'Comments' column found in intent_df")
else:
    print("✗ 'Comments' column NOT found in intent_df")
    print(f"Available columns: {list(intent_df.columns)}")

# Check Comment column in sentiment_df
if 'Comment' in sentiment_df.columns:
    print("✓ 'Comment' column found in sentiment_df")
else:
    print("✗ 'Comment' column NOT found in sentiment_df")
    print(f"Available columns: {list(sentiment_df.columns)}")

# Check for the target columns to merge
if 'Sentiment over comment from models' in sentiment_df.columns:
    print("✓ 'Sentiment over comment from models' column found in sentiment_df")
else:
    print("✗ 'Sentiment over comment from models' column NOT found in sentiment_df")

if 'final_label_three_models_final' in intent_df.columns:
    print("✓ 'final_label_three_models_final' column found in intent_df")
else:
    print("✗ 'final_label_three_models_final' column NOT found in intent_df")

print("\n" + "="*60)
print("DATA SAMPLE PREVIEW:")
print("="*60)
print("\nMain DF sample:")
print(main_df.head(2))
print("\nSentiment DF sample:")
print(sentiment_df.head(2))
print("\nIntent DF sample:")
print(intent_df.head(2))

Main DataFrame Info:
Shape: (3197, 13)
Columns: ['parent_id', 'id', 'author_id', 'time', 'title', 'text', 'post_url', 'reactions', 'year', 'nu_words', 'ha_id', 'text_ha', 'author_name']

Sentiment DataFrame Info:
Shape: (3197, 12)
Columns: ['Comment', 'Sentiment of the four models', 'Score', 'Overall voted sentiment', 'Overall Sentiment over comment from sentences', 'Sentiment from models', 'Sentiment from models score', 'Sentiment over comment from models', 'Score for entire comment', 'Post', 'Author Name', 'Comments_time']

Intent DataFrame Info:
Shape: (3197, 30)
Columns: ['Comments', 'GRO_NLP', 'roberta', 'debertaV3', 'GRO_NLP_logit_Appreciation', 'GRO_NLP_prob_Appreciation', 'GRO_NLP_logit_Criticism', 'GRO_NLP_prob_Criticism', 'GRO_NLP_logit_Inquiry', 'GRO_NLP_prob_Inquiry', 'GRO_NLP_logit_Statement', 'GRO_NLP_prob_Statement', 'roberta_logit_Appreciation', 'roberta_prob_Appreciation', 'roberta_logit_Criticism', 'roberta_prob_Criticism', 'roberta_logit_Inquiry', 'roberta_prob_Inqui

In [6]:
# Perform the merge operations
print("PERFORMING MERGES:")
print("="*60)

# First, let's check what text columns actually exist in the dataframes
print("Checking for text-like columns in each dataframe...")

# Check columns that might contain text/comments
def find_text_columns(df, df_name):
    text_cols = []
    for col in df.columns:
        if any(keyword in col.lower() for keyword in ['text', 'comment', 'content', 'message']):
            text_cols.append(col)
    print(f"{df_name} text-like columns: {text_cols}")
    return text_cols

main_text_cols = find_text_columns(main_df, "Main DF")
sentiment_text_cols = find_text_columns(sentiment_df, "Sentiment DF")
intent_text_cols = find_text_columns(intent_df, "Intent DF")

# Start with main_df as the base
merged_df = main_df.copy()

# Determine the correct column names for merging
# Try to find the most likely text column from main_df
if main_text_cols:
    main_text_col = main_text_cols[0]  # Use the first text-like column
    print(f"\nUsing '{main_text_col}' as main text column")
else:
    # Fallback to 'text' if it exists, otherwise use first column with string data
    main_text_col = 'text' if 'text' in main_df.columns else main_df.select_dtypes(include=['object']).columns[0]
    print(f"\nUsing '{main_text_col}' as fallback main text column")

# Merge with sentiment_df
if sentiment_text_cols:
    sentiment_text_col = sentiment_text_cols[0]
else:
    sentiment_text_col = 'Comment' if 'Comment' in sentiment_df.columns else sentiment_df.select_dtypes(include=['object']).columns[0]

print(f"Merging sentiment data using column: '{sentiment_text_col}'")

# Perform sentiment merge
if 'Sentiment over comment from models' in sentiment_df.columns:
    sentiment_merge_df = sentiment_df[[sentiment_text_col, 'Sentiment over comment from models']].copy()
    merged_df = merged_df.merge(
        sentiment_merge_df, 
        left_on=main_text_col, 
        right_on=sentiment_text_col, 
        how='left'
    )
    print(f"✓ Added sentiment data. New shape: {merged_df.shape}")
else:
    print("✗ Sentiment column not found, skipping sentiment merge")

# Merge with intent_df
if intent_text_cols:
    intent_text_col = intent_text_cols[0]
else:
    intent_text_col = 'Comments' if 'Comments' in intent_df.columns else intent_df.select_dtypes(include=['object']).columns[0]

print(f"Merging intent data using column: '{intent_text_col}'")

# Perform intent merge
if 'final_label_three_models_final' in intent_df.columns:
    intent_merge_df = intent_df[[intent_text_col, 'final_label_three_models_final']].copy()
    merged_df = merged_df.merge(
        intent_merge_df, 
        left_on=main_text_col, 
        right_on=intent_text_col, 
        how='left'
    )
    print(f"✓ Added intent data. New shape: {merged_df.shape}")
else:
    print("✗ Intent column not found, skipping intent merge")

# Clean up duplicate text columns if they exist
columns_to_drop = []
if sentiment_text_col in merged_df.columns and sentiment_text_col != main_text_col:
    columns_to_drop.append(sentiment_text_col)
if intent_text_col in merged_df.columns and intent_text_col != main_text_col:
    columns_to_drop.append(intent_text_col)

if columns_to_drop:
    merged_df = merged_df.drop(columns=columns_to_drop)
    print(f"✓ Cleaned up duplicate columns: {columns_to_drop}")

print(f"\nFinal merged dataframe shape: {merged_df.shape}")
print(f"Final columns: {list(merged_df.columns)}")

# Show sample of merged data
print("\n" + "="*60)
print("MERGED DATA SAMPLE:")
print("="*60)
target_cols = [main_text_col]
if 'Sentiment over comment from models' in merged_df.columns:
    target_cols.append('Sentiment over comment from models')
if 'final_label_three_models_final' in merged_df.columns:
    target_cols.append('final_label_three_models_final')

print(merged_df[target_cols].head())

# Check for successful merges
sentiment_success = merged_df['Sentiment over comment from models'].notna().sum() if 'Sentiment over comment from models' in merged_df.columns else 0
intent_success = merged_df['final_label_three_models_final'].notna().sum() if 'final_label_three_models_final' in merged_df.columns else 0

print(f"\nMerge Statistics:")
print(f"Sentiment matches: {sentiment_success}/{len(merged_df)} ({sentiment_success/len(merged_df)*100:.1f}%)")
print(f"Intent matches: {intent_success}/{len(merged_df)} ({intent_success/len(merged_df)*100:.1f}%)")

PERFORMING MERGES:
Checking for text-like columns in each dataframe...
Main DF text-like columns: ['text', 'text_ha']
Sentiment DF text-like columns: ['Comment', 'Overall Sentiment over comment from sentences', 'Sentiment over comment from models', 'Score for entire comment', 'Comments_time']
Intent DF text-like columns: ['Comments']

Using 'text' as main text column
Merging sentiment data using column: 'Comment'
✓ Added sentiment data. New shape: (41171, 15)
Merging intent data using column: 'Comments'
✓ Added intent data. New shape: (7119071, 17)
✓ Cleaned up duplicate columns: ['Comment', 'Comments']

Final merged dataframe shape: (7119071, 15)
Final columns: ['parent_id', 'id', 'author_id', 'time', 'title', 'text', 'post_url', 'reactions', 'year', 'nu_words', 'ha_id', 'text_ha', 'author_name', 'Sentiment over comment from models', 'final_label_three_models_final']

MERGED DATA SAMPLE:
                                                text  \
0  hoeveel KWh hebben de zonnepanelen op d

In [7]:
# Investigate why rows increased dramatically
print("INVESTIGATING ROW INCREASE:")
print("="*60)

print(f"Original main_df shape: {main_df.shape}")
print(f"Sentiment_df shape: {sentiment_df.shape}")
print(f"Intent_df shape: {intent_df.shape}")
print(f"Final merged_df shape: {merged_df.shape}")

print(f"\nRow increase: {merged_df.shape[0] - main_df.shape[0]:,} additional rows")
print(f"Multiplication factor: {merged_df.shape[0] / main_df.shape[0]:.1f}x")

# Check for duplicate keys that could cause Cartesian product
print("\n" + "="*60)
print("CHECKING FOR DUPLICATES IN TEXT COLUMNS:")
print("="*60)

print(f"\nMain DF '{main_text_col}' column:")
main_unique = main_df[main_text_col].nunique()
main_total = len(main_df)
print(f"  Total rows: {main_total:,}")
print(f"  Unique values: {main_unique:,}")
print(f"  Duplicates: {main_total - main_unique:,}")

print(f"\nSentiment DF '{sentiment_text_col}' column:")
sent_unique = sentiment_df[sentiment_text_col].nunique()
sent_total = len(sentiment_df)
print(f"  Total rows: {sent_total:,}")
print(f"  Unique values: {sent_unique:,}")
print(f"  Duplicates: {sent_total - sent_unique:,}")

print(f"\nIntent DF '{intent_text_col}' column:")
intent_unique = intent_df[intent_text_col].nunique()
intent_total = len(intent_df)
print(f"  Total rows: {intent_total:,}")
print(f"  Unique values: {intent_unique:,}")
print(f"  Duplicates: {intent_total - intent_unique:,}")

# Check for common values between dataframes
print("\n" + "="*60)
print("CHECKING OVERLAP BETWEEN DATAFRAMES:")
print("="*60)

main_set = set(main_df[main_text_col].astype(str))
sentiment_set = set(sentiment_df[sentiment_text_col].astype(str))
intent_set = set(intent_df[intent_text_col].astype(str))

print(f"Main DF unique texts: {len(main_set):,}")
print(f"Sentiment DF unique texts: {len(sentiment_set):,}")
print(f"Intent DF unique texts: {len(intent_set):,}")

main_sentiment_overlap = len(main_set.intersection(sentiment_set))
main_intent_overlap = len(main_set.intersection(intent_set))

print(f"\nOverlap between Main and Sentiment: {main_sentiment_overlap:,}")
print(f"Overlap between Main and Intent: {main_intent_overlap:,}")

# Show examples of duplicate text values
print("\n" + "="*60)
print("EXAMPLES OF DUPLICATE TEXT VALUES:")
print("="*60)

# Find text values that appear multiple times in intent_df
intent_duplicates = intent_df[intent_df.duplicated(subset=[intent_text_col], keep=False)].sort_values(intent_text_col)
if len(intent_duplicates) > 0:
    print(f"\nIntent DF duplicates (showing first 10):")
    print(intent_duplicates[[intent_text_col, 'final_label_three_models_final']].head(10))
    
    # Show how many times the most duplicated text appears
    most_duplicated = intent_df[intent_text_col].value_counts().head(5)
    print(f"\nMost duplicated texts in intent_df:")
    print(most_duplicated)
else:
    print("No duplicates found in intent_df")

INVESTIGATING ROW INCREASE:
Original main_df shape: (3197, 13)
Sentiment_df shape: (3197, 12)
Intent_df shape: (3197, 30)
Final merged_df shape: (7119071, 15)

Row increase: 7,115,874 additional rows
Multiplication factor: 2226.8x

CHECKING FOR DUPLICATES IN TEXT COLUMNS:

Main DF 'text' column:
  Total rows: 3,197
  Unique values: 2,930
  Duplicates: 267

Sentiment DF 'Comment' column:
  Total rows: 3,197
  Unique values: 2,930
  Duplicates: 267

Intent DF 'Comments' column:
  Total rows: 3,197
  Unique values: 2,930
  Duplicates: 267

CHECKING OVERLAP BETWEEN DATAFRAMES:
Main DF unique texts: 2,930
Sentiment DF unique texts: 2,930
Intent DF unique texts: 2,930

Overlap between Main and Sentiment: 2,930
Overlap between Main and Intent: 2,930

EXAMPLES OF DUPLICATE TEXT VALUES:

Intent DF duplicates (showing first 10):
      Comments final_label_three_models_final
45    <PERSON>                        Forward
1710  <PERSON>                        Forward
1711  <PERSON>                 

In [8]:
# Fix the merge by handling duplicates properly
print("FIXING THE MERGE PROBLEM:")
print("="*60)

# Strategy 1: Remove duplicates from intent_df, keeping the first occurrence
print("Strategy 1: Remove duplicates from intent_df")
intent_df_dedupe = intent_df.drop_duplicates(subset=[intent_text_col], keep='first')
print(f"Original intent_df: {len(intent_df):,} rows")
print(f"After removing duplicates: {len(intent_df_dedupe):,} rows")

# Strategy 2: Group by text and aggregate (e.g., take mode of labels)
print("\nStrategy 2: Aggregate duplicates (take most common label)")
intent_df_agg = intent_df.groupby(intent_text_col)['final_label_three_models_final'].agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0]).reset_index()
print(f"After aggregation: {len(intent_df_agg):,} rows")

# Let's go with Strategy 1 and redo the merge properly
print("\n" + "="*60)
print("REDOING THE MERGE WITH DEDUPLICATED DATA:")
print("="*60)

# Start fresh with main_df
corrected_df = main_df.copy()

# Merge with sentiment_df (this was already working correctly)
sentiment_merge_df = sentiment_df[[sentiment_text_col, 'Sentiment over comment from models']].copy()
corrected_df = corrected_df.merge(
    sentiment_merge_df, 
    left_on=main_text_col, 
    right_on=sentiment_text_col, 
    how='left'
)
print(f"After sentiment merge: {corrected_df.shape}")

# Merge with deduplicated intent_df
intent_merge_df_dedupe = intent_df_dedupe[[intent_text_col, 'final_label_three_models_final']].copy()
corrected_df = corrected_df.merge(
    intent_merge_df_dedupe, 
    left_on=main_text_col, 
    right_on=intent_text_col, 
    how='left'
)
print(f"After intent merge (deduplicated): {corrected_df.shape}")

# Clean up duplicate columns
columns_to_drop = []
if sentiment_text_col in corrected_df.columns and sentiment_text_col != main_text_col:
    columns_to_drop.append(sentiment_text_col)
if intent_text_col in corrected_df.columns and intent_text_col != main_text_col:
    columns_to_drop.append(intent_text_col)

if columns_to_drop:
    corrected_df = corrected_df.drop(columns=columns_to_drop)

print(f"Final corrected shape: {corrected_df.shape}")

# Verify the merge success
sentiment_matches = corrected_df['Sentiment over comment from models'].notna().sum()
intent_matches = corrected_df['final_label_three_models_final'].notna().sum()

print(f"\nMerge success rates:")
print(f"Sentiment matches: {sentiment_matches}/{len(corrected_df)} ({sentiment_matches/len(corrected_df)*100:.1f}%)")
print(f"Intent matches: {intent_matches}/{len(corrected_df)} ({intent_matches/len(corrected_df)*100:.1f}%)")

# Replace the problematic merged_df with the corrected one
merged_df = corrected_df
print(f"\n✓ Updated merged_df with corrected data")
print(f"New merged_df shape: {merged_df.shape}")

FIXING THE MERGE PROBLEM:
Strategy 1: Remove duplicates from intent_df
Original intent_df: 3,197 rows
After removing duplicates: 2,930 rows

Strategy 2: Aggregate duplicates (take most common label)
After aggregation: 2,930 rows

REDOING THE MERGE WITH DEDUPLICATED DATA:
After sentiment merge: (41171, 15)
After intent merge (deduplicated): (41171, 17)
Final corrected shape: (41171, 15)

Merge success rates:
Sentiment matches: 41171/41171 (100.0%)
Intent matches: 41171/41171 (100.0%)

✓ Updated merged_df with corrected data
New merged_df shape: (41171, 15)


In [9]:
# Investigate why sentiment merge also multiplies rows
print("INVESTIGATING SENTIMENT MERGE ISSUE:")
print("="*60)

print(f"Main DF shape: {main_df.shape}")
print(f"Sentiment DF shape: {sentiment_df.shape}")

# Check for duplicates in both main_df and sentiment_df
main_text_duplicates = main_df[main_df.duplicated(subset=[main_text_col], keep=False)]
sentiment_text_duplicates = sentiment_df[sentiment_df.duplicated(subset=[sentiment_text_col], keep=False)]

print(f"\nDuplicates in main_df text column: {len(main_text_duplicates)}")
print(f"Duplicates in sentiment_df text column: {len(sentiment_text_duplicates)}")

if len(main_text_duplicates) > 0:
    print(f"\nMain DF duplicate examples:")
    print(main_text_duplicates[[main_text_col]].head())

if len(sentiment_text_duplicates) > 0:
    print(f"\nSentiment DF duplicate examples:")
    print(sentiment_text_duplicates[[sentiment_text_col, 'Sentiment over comment from models']].head())

# Test simple merge to see what happens
print("\nTesting simple merge main_df + sentiment_df:")
test_merge = main_df.merge(
    sentiment_df[[sentiment_text_col, 'Sentiment over comment from models']], 
    left_on=main_text_col, 
    right_on=sentiment_text_col, 
    how='left'
)
print(f"Result shape: {test_merge.shape}")
print(f"Multiplication factor: {test_merge.shape[0] / main_df.shape[0]:.1f}x")

# The solution: We need to treat this as exact row matching since both dataframes 
# are the same size and should correspond row by row
print("\n" + "="*60)
print("PROPER FIX: ROW-BY-ROW ALIGNMENT")
print("="*60)

# Check if the dataframes are actually aligned (same order)
print("Checking if dataframes are row-aligned...")
if len(main_df) == len(sentiment_df) == len(intent_df):
    print("✓ All dataframes have the same number of rows")
    print("✓ Assuming they are row-aligned, we can use index-based assignment")
    
    # Create final dataframe using index alignment instead of text-based merging
    final_df = main_df.copy()
    
    # Add sentiment column by index
    final_df['Sentiment over comment from models'] = sentiment_df['Sentiment over comment from models'].values
    
    # Add intent column by index (after deduplication)
    # We need to map intent data properly
    intent_df_dedupe = intent_df.drop_duplicates(subset=[intent_text_col], keep='first')
    
    # Create a mapping from text to intent
    intent_mapping = dict(zip(intent_df_dedupe[intent_text_col], intent_df_dedupe['final_label_three_models_final']))
    
    # Map intent labels to main_df
    final_df['final_label_three_models_final'] = final_df[main_text_col].map(intent_mapping)
    
    print(f"\nFinal dataframe shape: {final_df.shape}")
    print(f"✓ Maintained original row count!")
    
    # Check success rates
    sentiment_success = final_df['Sentiment over comment from models'].notna().sum()
    intent_success = final_df['final_label_three_models_final'].notna().sum()
    
    print(f"\nSuccess rates:")
    print(f"Sentiment data: {sentiment_success}/{len(final_df)} ({sentiment_success/len(final_df)*100:.1f}%)")
    print(f"Intent data: {intent_success}/{len(final_df)} ({intent_success/len(final_df)*100:.1f}%)")
    
    # Update the merged_df variable
    merged_df = final_df
    print(f"\n✓ Updated merged_df with correct row count")
    
else:
    print("✗ Dataframes have different sizes - cannot use index alignment")
    print(f"Main DF: {len(main_df)}, Sentiment DF: {len(sentiment_df)}, Intent DF: {len(intent_df)}")

print(f"\nFinal merged_df shape: {merged_df.shape}")

INVESTIGATING SENTIMENT MERGE ISSUE:
Main DF shape: (3197, 13)
Sentiment DF shape: (3197, 12)

Duplicates in main_df text column: 292
Duplicates in sentiment_df text column: 292

Main DF duplicate examples:
         text
45   <PERSON>
56   <PERSON>
99   <PERSON>
109  <PERSON>
128  <PERSON>

Sentiment DF duplicate examples:
      Comment  Sentiment over comment from models
45   <PERSON>                                   0
56   <PERSON>                                   0
99   <PERSON>                                   0
109  <PERSON>                                   0
128  <PERSON>                                   0

Testing simple merge main_df + sentiment_df:
Result shape: (41171, 15)
Multiplication factor: 12.9x

PROPER FIX: ROW-BY-ROW ALIGNMENT
Checking if dataframes are row-aligned...
✓ All dataframes have the same number of rows
✓ Assuming they are row-aligned, we can use index-based assignment

Final dataframe shape: (3197, 15)
✓ Maintained original row count!

Success rates:
S

In [10]:
# Final verification and summary
print("FINAL MERGE VERIFICATION:")
print("="*60)

print(f"Original main_df shape: {main_df.shape}")
print(f"Final merged_df shape: {merged_df.shape}")
print(f"✓ Row count preserved: {merged_df.shape[0] == main_df.shape[0]}")

print(f"\nColumns in merged_df:")
for i, col in enumerate(merged_df.columns, 1):
    print(f"  {i}. {col}")

print(f"\nSample of merged data:")
print(merged_df[['text', 'Sentiment over comment from models', 'final_label_three_models_final']].head())

print(f"\nData completeness:")
print(f"- Sentiment data: {merged_df['Sentiment over comment from models'].notna().sum()}/{len(merged_df)} (100.0%)")
print(f"- Intent data: {merged_df['final_label_three_models_final'].notna().sum()}/{len(merged_df)} (100.0%)")

print("\n✅ MERGE COMPLETED SUCCESSFULLY!")
print("✅ All data preserved with correct row count!")

FINAL MERGE VERIFICATION:
Original main_df shape: (3197, 13)
Final merged_df shape: (3197, 15)
✓ Row count preserved: True

Columns in merged_df:
  1. parent_id
  2. id
  3. author_id
  4. time
  5. title
  6. text
  7. post_url
  8. reactions
  9. year
  10. nu_words
  11. ha_id
  12. text_ha
  13. author_name
  14. Sentiment over comment from models
  15. final_label_three_models_final

Sample of merged data:
                                                text  \
0  hoeveel KWh hebben de zonnepanelen op de wezen...   
1  Dan kom ik langs voor sleutels en krijg er gra...   
2  Mmmm, iets nieuws? Wij hebben in maart geen wi...   
3  Het zou beter zijn als Samenwerking een aantal...   
4  Leuk als je in een woning woont met warmtepomp...   

   Sentiment over comment from models final_label_three_models_final  
0                                  -1                        Inquiry  
1                                   0                      Criticism  
2                                  

In [14]:
merged_df.rename(columns={'Sentiment over comment from models': 'Sentiment',
                          'final_label_three_models_final': 'Intent'}, inplace=True)

In [15]:
merged_df.to_csv(r"C:\Users\20245179\OneDrive - TU Eindhoven\WP2_Simulation_Tool\data\final_merged_data.csv", index=False)

In [18]:
import pandas as pd
import  numpy as np
df_model = pd.read_csv(r"C:\Users\20245179\OneDrive - TU Eindhoven\WP2_Simulation_Tool\data\final_merged_data.csv")

# IMPORTANT FIX: Use 'Post' column for post features, not 'Comment' column
df_model['text_ha'] = df_model['text_ha'].astype(str)

print("🔧 Creating post characteristics features from POST content...")
print("⚠️  CORRECTION: Now using 'Post' column (actual posts) instead of 'Comment' column")

# 1. Post length (word count) - FROM THE ACTUAL POST
df_model['post_length'] = df_model['text_ha'].apply(lambda x: len(str(x).split()))

# 2. Length categories - BASED ON POST LENGTH
bins = [0, 50, 150, 300, np.inf]
labels = ["tweet (<50)", "short (50-150)", "abstract (150-300)", "long (>300)"]
df_model["length_category"] = pd.cut(df_model["post_length"], bins=bins, labels=labels,
                                    right=True, include_lowest=True)

# 3. Number of unique words - FROM THE ACTUAL POST
df_model['unique_words'] = df_model['text_ha'].apply(lambda x: len(set(str(x).split())))

# 4. Unique words percentage - BASED ON POST CONTENT
df_model['unique_words_ratio'] = ((df_model['unique_words'] / df_model['post_length']) * 100).round(2)
# Handle division by zero
df_model['unique_words_ratio'] = df_model['unique_words_ratio'].fillna(0)

# 5. Question present (binary) - QUESTIONS IN THE POST
df_model["has_questions"] = df_model["text_ha"].str.contains(r"\?", regex=True).astype(int)

# 6. URL present (binary) - URLS IN THE POST
df_model["has_urls"] = df_model["text_ha"].str.contains(r"http\S+", regex=True).astype(int)

🔧 Creating post characteristics features from POST content...
⚠️  CORRECTION: Now using 'Post' column (actual posts) instead of 'Comment' column


In [19]:
df_model

,parent_id,id,author_id,time,title,text,post_url,reactions,year,nu_words,...,text_ha,author_name,Sentiment,Intent,post_length,length_category,unique_words,unique_words_ratio,has_questions,has_urls
0,ZmVlZGJhY2s6MjA0MDM2MDg0NjAxMzE1NA==,ZmVlZGJhY2s6MjA0MDM2MDg0NjAxMzE1NF8yMDQxMDI4Nj...,100001659249038,2019-04-02T22:14:59,NaN,hoeveel KWh hebben de zonnepanelen op de wezen...,0,0,2019,10,...,De zonnepanelen op het dak van ons kantoor doe...,WormerWonen,-1,Inquiry,65,short (50-150),49,75.38,0,0
1,ZmVlZGJhY2s6MjA0ODM5MDA5ODU1MTgzMw==,ZmVlZGJhY2s6MjA0ODM5MDA5ODU1MTgzM18yMDc5MTcxNj...,1204819098,2018-12-21T15:37:26,NaN,Dan kom ik langs voor sleutels en krijg er gra...,0,0,2018,17,...,IN ACTI-UM | Onze vakmannen vervangen bij iede...,Actium,0,Criticism,61,short (50-150),47,77.05,0,0
2,ZmVlZGJhY2s6MjA0ODM5MDA5ODU1MTgzMw==,ZmVlZGJhY2s6MjA0ODM5MDA5ODU1MTgzM18yMDc4ODYzNj...,627271250,2018-12-21T10:57:48,NaN,"Mmmm, iets nieuws? Wij hebben in maart geen wi...",0,0,2018,10,...,IN ACTI-UM | Onze vakmannen vervangen bij iede...,Actium,-1,Statement,61,short (50-150),47,77.05,0,0
3,ZmVlZGJhY2s6MjA1MDEwNjkzMTY4ODQzNA==,ZmVlZGJhY2s6MjA1MDEwNjkzMTY4ODQzNF8yMDc1OTI1Mj...,100001478616616,2018-08-01T15:22:31,NaN,Het zou beter zijn als Samenwerking een aantal...,0,0,2018,23,...,Een hoog energieverbruik is slecht voor het mi...,NaN,-1,Statement,51,short (50-150),48,94.12,1,1
4,ZmVlZGJhY2s6MjA1MDEwNjkzMTY4ODQzNA==,ZmVlZGJhY2s6MjA1MDEwNjkzMTY4ODQzNF8yMDczOTE5Nj...,100000855817347,2018-07-31T11:51:36,NaN,Leuk als je in een woning woont met warmtepomp...,0,0,2018,34,...,Een hoog energieverbruik is slecht voor het mi...,NaN,0,Criticism,51,short (50-150),48,94.12,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3192,ZmVlZGJhY2s6OTU1MjE0NDg0ODMyNDYw,ZmVlZGJhY2s6OTU1MjE0NDg0ODMyNDYwXzk1NTIyMDUzND...,100001182464975,2019-07-30T10:57:55,NaN,Kijk onze wijk <PERSON> <PERSON> Paulo Mariaa ...,0,2,2019,20,...,Vorige week zijn de laatste huizen van 't Getf...,De Woonplaats,0,Statement,46,tweet (<50),37,80.43,0,0
3193,ZmVlZGJhY2s6OTYwNDYyMjc0MzY0NTIx,ZmVlZGJhY2s6OTYwNDYyMjc0MzY0NTIxXzk2NjA4NTQ4Nz...,596212634,2020-10-22T12:40:07,NaN,Is er genoeg ventilatie en / of kunnen ramen g...,0,0,2020,10,...,Op de 1e verdieping van het oude schoolgebouw ...,Woongoed Zeist,0,Inquiry,62,short (50-150),53,85.48,0,0
3194,ZmVlZGJhY2s6OTYwNDYyMjc0MzY0NTIx,ZmVlZGJhY2s6OTYwNDYyMjc0MzY0NTIxXzk2NTQwMTMyNz...,100001224379149,2020-10-21T15:54:40,NaN,<PERSON> kan je dus straks gebruik van maken. ...,0,0,2020,21,...,Op de 1e verdieping van het oude schoolgebouw ...,Woongoed Zeist,0,Forward,62,short (50-150),53,85.48,0,0
3195,ZmVlZGJhY2s6OTYwNDYyMjc0MzY0NTIx,ZmVlZGJhY2s6OTYwNDYyMjc0MzY0NTIxXzk2MzU0MDgwNz...,100002916564091,2020-10-19T10:14:18,NaN,Komen er nog meer werkplekken bij? Of blijft h...,0,0,2020,16,...,Op de 1e verdieping van het oude schoolgebouw ...,Woongoed Zeist,-1,Inquiry,62,short (50-150),53,85.48,0,0


In [20]:
df_model.to_csv(r"C:\Users\20245179\OneDrive - TU Eindhoven\WP2_Simulation_Tool\data\final_merged_data.csv")

In [3]:
def analyze_correlations(csv_path: str):
    """
    Analyze correlation between POST features and COMMENT characteristics.
    Find which post feature best predicts comment length/style.
    """
    import pandas as pd
    import numpy as np
    from scipy.stats import chi2_contingency, pointbiserialr, spearmanr
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    df = pd.read_csv(csv_path)
    df = df.rename(columns={"text_ha": "post", "text": "comment"})
    
    # Calculate comment features
    df['comment_word_count'] = df['comment'].str.split().str.len()
    df['comment_length_cat'] = pd.cut(
        df['comment_word_count'],
        bins=[0, 10, 25, 50, float('inf')],
        labels=['zeer_kort', 'kort', 'gemiddeld', 'lang']
    )
    
    print("\n" + "="*80)
    print("CORRELATION ANALYSIS: POST FEATURES → COMMENT CHARACTERISTICS")
    print("="*80)
    
    # ============================================
    # 1. POST LENGTH_CATEGORY → COMMENT LENGTH
    # ============================================
    print("\n" + "="*80)
    print("1. POST LENGTH_CATEGORY → COMMENT LENGTH")
    print("="*80)
    
    crosstab = pd.crosstab(df['length_category'], df['comment_length_cat'], normalize='index')
    print("\nConditional distribution P(comment_length | post_length):")
    print(crosstab.round(3))
    
    # Chi-square test
    contingency_table = pd.crosstab(df['length_category'], df['comment_length_cat'])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    
    # Cramér's V (effect size for categorical variables)
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape[0], contingency_table.shape[1]) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim))
    
    print(f"\nChi-square test:")
    print(f"  χ² = {chi2:.2f}, p-value = {p_value:.4f}")
    print(f"  Cramér's V = {cramers_v:.3f}")
    print(f"  Interpretation: {interpret_cramers_v(cramers_v)}")
    
    # ============================================
    # 2. UNIQUE_WORDS_RATIO → COMMENT LENGTH
    # ============================================
    print("\n" + "="*80)
    print("2. UNIQUE_WORDS_RATIO (POST) → COMMENT LENGTH")
    print("="*80)
    
    # Binned analysis
    df['unique_words_binned'] = pd.cut(
        df['unique_words_ratio'],
        bins=[0, 40, 60, 100],
        labels=['low (<40%)', 'medium (40-60%)', 'high (>60%)']
    )
    
    crosstab_unique = pd.crosstab(df['unique_words_binned'], df['comment_length_cat'], normalize='index')
    print("\nConditional distribution P(comment_length | unique_words_ratio):")
    print(crosstab_unique.round(3))
    
    # Chi-square test
    contingency_unique = pd.crosstab(df['unique_words_binned'], df['comment_length_cat'])
    chi2_u, p_value_u, dof_u, expected_u = chi2_contingency(contingency_unique)
    n_u = contingency_unique.sum().sum()
    min_dim_u = min(contingency_unique.shape[0], contingency_unique.shape[1]) - 1
    cramers_v_u = np.sqrt(chi2_u / (n_u * min_dim_u))
    
    print(f"\nChi-square test:")
    print(f"  χ² = {chi2_u:.2f}, p-value = {p_value_u:.4f}")
    print(f"  Cramér's V = {cramers_v_u:.3f}")
    print(f"  Interpretation: {interpret_cramers_v(cramers_v_u)}")
    
    # Continuous correlation
    # Encode comment_length_cat as ordinal
    length_order = {'zeer_kort': 1, 'kort': 2, 'gemiddeld': 3, 'lang': 4}
    df['comment_length_ordinal'] = df['comment_length_cat'].map(length_order)
    
    spearman_corr, spearman_p = spearmanr(df['unique_words_ratio'], df['comment_length_ordinal'])
    print(f"\nSpearman correlation (continuous):")
    print(f"  ρ = {spearman_corr:.3f}, p-value = {spearman_p:.4f}")
    print(f"  Interpretation: {interpret_correlation(spearman_corr)}")
    
    # ============================================
    # 3. HAS_QUESTIONS → COMMENT LENGTH
    # ============================================
    print("\n" + "="*80)
    print("3. HAS_QUESTIONS (POST) → COMMENT LENGTH")
    print("="*80)
    
    crosstab_q = pd.crosstab(df['has_questions'], df['comment_length_cat'], normalize='index')
    print("\nConditional distribution P(comment_length | has_questions):")
    print(crosstab_q.round(3))
    
    # Chi-square test
    contingency_q = pd.crosstab(df['has_questions'], df['comment_length_cat'])
    chi2_q, p_value_q, dof_q, expected_q = chi2_contingency(contingency_q)
    n_q = contingency_q.sum().sum()
    min_dim_q = min(contingency_q.shape[0], contingency_q.shape[1]) - 1
    cramers_v_q = np.sqrt(chi2_q / (n_q * min_dim_q))
    
    print(f"\nChi-square test:")
    print(f"  χ² = {chi2_q:.2f}, p-value = {p_value_q:.4f}")
    print(f"  Cramér's V = {cramers_v_q:.3f}")
    print(f"  Interpretation: {interpret_cramers_v(cramers_v_q)}")
    
    # Point-biserial correlation (binary → continuous)
    point_biserial_corr, point_biserial_p = pointbiserialr(
        df['has_questions'], 
        df['comment_word_count']
    )
    print(f"\nPoint-biserial correlation (binary → word count):")
    print(f"  r = {point_biserial_corr:.3f}, p-value = {point_biserial_p:.4f}")
    print(f"  Interpretation: {interpret_correlation(point_biserial_corr)}")
    
    # ============================================
    # 4. HAS_URLS → COMMENT LENGTH
    # ============================================
    print("\n" + "="*80)
    print("4. HAS_URLS (POST) → COMMENT LENGTH")
    print("="*80)
    
    crosstab_url = pd.crosstab(df['has_urls'], df['comment_length_cat'], normalize='index')
    print("\nConditional distribution P(comment_length | has_urls):")
    print(crosstab_url.round(3))
    
    # Chi-square test
    contingency_url = pd.crosstab(df['has_urls'], df['comment_length_cat'])
    chi2_url, p_value_url, dof_url, expected_url = chi2_contingency(contingency_url)
    n_url = contingency_url.sum().sum()
    min_dim_url = min(contingency_url.shape[0], contingency_url.shape[1]) - 1
    cramers_v_url = np.sqrt(chi2_url / (n_url * min_dim_url))
    
    print(f"\nChi-square test:")
    print(f"  χ² = {chi2_url:.2f}, p-value = {p_value_url:.4f}")
    print(f"  Cramér's V = {cramers_v_url:.3f}")
    print(f"  Interpretation: {interpret_cramers_v(cramers_v_url)}")
    
    # Point-biserial correlation
    point_biserial_url, point_biserial_url_p = pointbiserialr(
        df['has_urls'], 
        df['comment_word_count']
    )
    print(f"\nPoint-biserial correlation (binary → word count):")
    print(f"  r = {point_biserial_url:.3f}, p-value = {point_biserial_url_p:.4f}")
    print(f"  Interpretation: {interpret_correlation(point_biserial_url)}")
    
    # ============================================
    # 5. SUMMARY COMPARISON
    # ============================================
    print("\n" + "="*80)
    print("SUMMARY: WHICH POST FEATURE BEST PREDICTS COMMENT LENGTH?")
    print("="*80)
    
    results = {
        'POST length_category': {
            'metric': 'Cramér\'s V',
            'value': cramers_v,
            'p_value': p_value
        },
        'POST unique_words_ratio': {
            'metric': 'Cramér\'s V',
            'value': cramers_v_u,
            'p_value': p_value_u
        },
        'POST has_questions': {
            'metric': 'Cramér\'s V',
            'value': cramers_v_q,
            'p_value': p_value_q
        },
        'POST has_urls': {
            'metric': 'Cramér\'s V',
            'value': cramers_v_url,
            'p_value': p_value_url
        },
    }
    
    print("\nRanked by effect size (Cramér's V):")
    print("-" * 60)
    sorted_results = sorted(results.items(), key=lambda x: x[1]['value'], reverse=True)
    
    for i, (feature, stats) in enumerate(sorted_results, 1):
        significance = "***" if stats['p_value'] < 0.001 else "**" if stats['p_value'] < 0.01 else "*" if stats['p_value'] < 0.05 else "ns"
        print(f"{i}. {feature:30s} | V = {stats['value']:.3f} | p = {stats['p_value']:.4f} {significance}")
    
    print("\nSignificance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")
    
    best_feature = sorted_results[0][0]
    best_value = sorted_results[0][1]['value']
    
    print(f"\n{'='*80}")
    print(f"🏆 WINNER: {best_feature}")
    print(f"   Effect size (Cramér's V) = {best_value:.3f}")
    print(f"   {interpret_cramers_v(best_value)}")
    print(f"{'='*80}")
    
    print("\n📊 RECOMMENDATION FOR STRATIFICATION:")
    if best_value < 0.1:
        print("   ⚠️  All correlations are WEAK (V < 0.1)")
        print("   → Consider using GLOBAL comment length distribution")
        print("   → Or use POST length_category as it's most interpretable")
    elif best_feature == 'POST length_category':
        print(f"   ✅ Use stratification: df['length_category']")
    elif best_feature == 'POST unique_words_ratio':
        print(f"   ✅ Use stratification: df['unique_words_binned']")
    elif best_feature == 'POST has_questions':
        print(f"   ✅ Use stratification: df['has_questions']")
    elif best_feature == 'POST has_urls':
        print(f"   ✅ Use stratification: df['has_urls']")
    
    return df, results


def interpret_cramers_v(v):
    """Interpret Cramér's V effect size."""
    if v < 0.1:
        return "Negligible association"
    elif v < 0.3:
        return "Weak association"
    elif v < 0.5:
        return "Moderate association"
    else:
        return "Strong association"


def interpret_correlation(r):
    """Interpret correlation coefficient."""
    abs_r = abs(r)
    if abs_r < 0.1:
        return "Negligible correlation"
    elif abs_r < 0.3:
        return "Weak correlation"
    elif abs_r < 0.5:
        return "Moderate correlation"
    else:
        return "Strong correlation"


# ============================================
# USAGE
# ============================================
if __name__ == "__main__":
    csv_path = r"C:\Users\20245179\OneDrive - TU Eindhoven\WP2_Simulation_Tool\data\final_merged_data.csv"  # Update this
    df, results = analyze_correlations(csv_path)





CORRELATION ANALYSIS: POST FEATURES → COMMENT CHARACTERISTICS

1. POST LENGTH_CATEGORY → COMMENT LENGTH

Conditional distribution P(comment_length | post_length):
comment_length_cat  zeer_kort   kort  gemiddeld   lang
length_category                                       
abstract (150-300)      0.412  0.245      0.202  0.142
long (>300)             0.500  0.500      0.000  0.000
short (50-150)          0.467  0.291      0.161  0.082
tweet (<50)             0.468  0.303      0.146  0.083

Chi-square test:
  χ² = 16.93, p-value = 0.0498
  Cramér's V = 0.042
  Interpretation: Negligible association

2. UNIQUE_WORDS_RATIO (POST) → COMMENT LENGTH

Conditional distribution P(comment_length | unique_words_ratio):
comment_length_cat   zeer_kort   kort  gemiddeld   lang
unique_words_binned                                    
medium (40-60%)          0.143  0.286      0.214  0.357
high (>60%)              0.465  0.291      0.159  0.085

Chi-square test:
  χ² = 15.33, p-value = 0.0016
  Cramér'